In [4]:
%pip install tinygrad
%pip install sympy

import tinygrad as tg
import numpy as np
from typing import Tuple, Literal
import sympy as sp


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from collections.abc import Callable, Iterable
from collections import OrderedDict


def find(iterator:Iterable, cond:Callable):
        for it in iter(iterator):
            if cond(it):
                yield element

def recursively_create_zeros(tensor):
        """
        Recursively loop through a tensor and convert its elements into zeros. 
        We use this when calculating the gradient vector between states of our tensor inside of computational graph.  
        """
        # Our terminal / base case.
        if not tensor.shape:
            return 0.0

        if len(tensor.shape) == 1:
            return [0.0 for _ in range(tensor.shape[0])]
        return [recursively_create_zeros(tensor.shape[1:]) for _ in range(tensor.shape[0])]


class CompGradGraph:
    """
    Directed acyclic graph consisting of function objects
    leaves are input tensors, roots are output tensors. 
    Chase graph from roots to leaves to automatically compute gradients using chain rule.
    """
    def __init__(self):
        self.nodes = {}
    

    def get_graph(self):
        g = []
        for node_id, node in sorted(self.nodes.items()):
            g.append((node_id, node))
        return g
        

    def add_node(self, node):
        if self.nodes is None:
            self.nodes = OrderedDict((node.id, node))
            
        self.nodes[node.id] = node
        print(self.nodes[node.id] == node, 'adding node')


    def add_directed_edges(self, edges:list):
        for edge_1, operation, edge_2 in edges:
            if edge_1 not in self.nodes: self.nodes[edge_1] = [operation]
            if edge_2 not in self.nodes: self.nodes[edge_2] = [operation]
            self.nodes[edge_1].append(edge_2)

__graph__ = CompGradGraph()

class Ops:
    ADD = 'add'
    MUL = 'mul' # element wise multiplication between vectors in tensor.
    SUB = 'sub' # element wise subtraction between vectors in tensor.
    MATMUL = 'matmul' # matrix multipication between tensors/
    DIV = 'div'

class Tensor:
    _next_id = 0

    def __init__(
        self,
        data,
        dtype:str = 'float32',
        req_grad:bool = False,
    ):
        Tensor._next_id += 1
        self.id = Tensor._next_id
        self.data = np.matrix(data, dtype=dtype)
        self.dtype = dtype
        self.req_grad = req_grad
        self.is_leaf = True
        self.grad_fn = None
        self.parents = []
        self.grad = None
        __graph__.add_node(self)

    def __add__(self, tensor):
        edges = []
        edges = [self.id, Ops.ADD, tensor.id]
        __graph__.add_directed_edges(edges)


    def calc_gradient(self, op, y):
        """
        Rehash on gradients.
        Gradients depend on a mathematical operation between two tensors. i.e.

        x = Tensor(2)
        y = x * 2

        gradient = 2

        ---

        x = Tensor(4,4)
        y = x * 2

        gradient = 2

        ---

        x = Tensor(2)
        y = x * 2 - x / 2
        f(x) = 2x - 1/2x = 3/2x

        gradient =  1.5

        ---

        x = Tensor(2, 2, 2)
        y = x + x^2 = 1 + 2x

        gradient = 5

        ---

        torch backwards() is called on the error tensor - 
        Autograd calculates + stores the gradients for each model parameter
        in the params .grad attribute. 

        You load your gradients (or model.parameters) into an optimizer - like SGD or ADAM
        Then you call optimizer.step() to actually initiate gradient descent (back propegation).

        All of this starts with being able to calculate a gradient between two tensors. This function 
        will serve as the basis for our tensor class. (not vibe coded this time)!

        self and y represent model parameters (weights or layers)
        loss tensor represents the error we are learning from.

        Remember - The gradient vector points towards the direction of steepest ascent /
        The steepest direction maximizes the directional derivative.
        """
        assert self.get_shape == y.get_shape
        
        if not op:
            return

        # They gotta be the same shape because we aren't fucking with broadcasting yet. 
        result = recursively_create_zeros(self.data)
        
        match op:
            #TODO: calc gradient for diff opps
            case Ops.MUL:
                # differentiating a division op for a tensor
                grad = self.differentiate_mul_op(y)
                return 
            case Ops.ADD:
                return
            case Ops.MATMUL:
                return
            case Ops.DIV:
                return

    def differentiate_mul_op(self, y):


    @property
    def get_shape(self):
        return self.data.shape

    def backwards(self):
        """Jacobian of self w.r.t. ref (column vectors)"""
        m, n = self.data.size, ref.data.size

        

In [21]:
t = Tensor(2)

print(__graph__.nodes)

t2 = Tensor(2)

True adding node
{1: <__main__.Tensor object at 0x110391390>, 2: <__main__.Tensor object at 0x1103904d0>}
True adding node


In [27]:

t2 = Tensor(3)

__graph__.__dict__

True adding node


{'nodes': {1: <__main__.Tensor at 0x1059a5d10>}}

In [31]:
g = __graph__.get_graph()
g[0][1]

In [23]:
# Independent numeric arrays — no symbols → zero Jacobian (no SymPy call)
t1 = Tensor(np.random.randn(10, 10))
t2 = Tensor(np.random.randn(10, 10))
t3 = Tensor(np.random.randn(10, 10))
# j = t2.calc_gradient(t1)

# Assert every instance gets the same gradient graph huzzah.
assert t1.gradient_graph == t2.gradient_graph == t3.gradient_graph
